# LLM Fundamentals: Tokens, Transformers, Decoding and the KV Cache

In this notebook you will: train a tiny BPE tokenizer, inspect positional encodings, verify that a KV cache
reproduces full recomputation exactly, compare decoding strategies, and use scaling-law arithmetic.
All components are tiny and CPU-only; no pretrained weights are used.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/llm-fundamentals/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import llm_core
print('llm_core loaded')

## 1. Byte-Pair Encoding
Merges are learned greedily from pair frequencies, then replayed in the same order at encode time.

In [ ]:
corpus = ['low low low lower lowest', 'new newer newest', 'lower newer']
merges = llm_core.train_bpe(corpus, 12)
print('first merges:', merges[:6])
for w in ['lowest', 'newest', 'zebra']:
    toks = llm_core.bpe_encode_word(w, merges)
    print(w, '->', toks, '->', llm_core.bpe_decode(toks))

## 2. Positional encoding
Sinusoids give each position a unique, smoothly varying code; nearby positions have higher dot-product similarity.

In [ ]:
pe = llm_core.sinusoidal_positions(64, 32)
sim = (pe @ pe.T)
print('sim(0,1)=%.2f  sim(0,8)=%.2f  sim(0,40)=%.2f' % (sim[0,1], sim[0,8], sim[0,40]))

## 3. KV cache: same answer, less work
Greedy decoding with and without the cache must give identical tokens; only the amount of recomputation differs.

In [ ]:
import torch
torch.manual_seed(0)
m = llm_core.TinyGPT(vocab=40, dim=32, heads=4, layers=2)
prompt = [3, 7, 11, 2]
a = llm_core.generate(m, prompt, 16, use_cache=True)
b = llm_core.generate(m, prompt, 16, use_cache=False)
print('identical outputs:', a == b)
without, with_cache = llm_core.kv_cache_matmul_savings(len(prompt), 16)
print(f'token-positions processed: {without} without cache vs {with_cache} with cache')
assert a == b

## 4. Decoding strategies
Temperature reshapes the distribution; top-k and top-p truncate its tail. Entropy shows how much randomness remains.

In [ ]:
import torch, math
logits = torch.tensor([3.0, 2.5, 1.0, 0.2, -1.0, -2.0])
def entropy(p): 
    p = p[p > 0]
    return float(-(p * p.log()).sum())
for name, kw in [('T=1', {}), ('T=0.5', dict(temperature=0.5)), ('T=2', dict(temperature=2.0)),
                 ('top_k=2', dict(top_k=2)), ('top_p=0.8', dict(top_p=0.8))]:
    p = llm_core.filter_logits(logits, **kw)
    print(f'{name:10s} support={(p>0).sum().item()} entropy={entropy(p):.3f} probs={[round(x,3) for x in p.tolist()]}')

## 5. Scaling-law arithmetic and context budgeting
The constants are an illustrative published fit (Hoffmann et al., 2022); they are not universal.

In [ ]:
for c in [1e21, 1e23, 1e25]:
    n, d = llm_core.compute_optimal(c)
    print(f'C={c:.0e} FLOPs -> N*={n:.2e} params, D*={d:.2e} tokens (D/N={d/n:.1f}), loss={llm_core.parametric_loss(n, d):.3f}')
toks = list(range(100))
print(llm_core.fit_to_context(toks, 10, keep_prefix=3))